# Laboratorio 2
### Jonathan Espinoza 20022
### Juan Fernando Ramirez 20666

#### Categorizaci?n de peliculas por genero utilizando redes neuronales. 

In [ ]:
import tensorflow as tf
import pandas as pd
import numpy as np
from scipy.stats import pointbiserialr
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, BatchNormalization
from tensorflow.keras.regularizers import l2, l1
from tensorflow.keras.layers import Dropout
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
df = pd.read_csv('movie_statistic_dataset.csv')
df.head()

In [ ]:
df['genres_list'] = df['genres'].str.split(',')

genres_dummies = df['genres_list'].explode().str.get_dummies().sum(level=0)
df_movies_dummies = pd.concat([df, genres_dummies], axis=1)
df_movies_dummies.drop(['genres_list', 'genres', "\\N"], axis=1, inplace=True)

df_movies_dummies.columns
df_movies_dummies.head()



In [ ]:
professions = df_movies_dummies['director_professions'].str.split(',', expand=True).stack().unique()
professions = sorted(professions)

professions.remove('-')

for profession in professions:
    df_movies_dummies[f'director_is_{profession}'] = df_movies_dummies['director_professions'].str.contains(profession).astype(int)

In [ ]:
df_movies_dummies['production_date'] = pd.to_datetime(df_movies_dummies['production_date'], errors='coerce')
df_movies_dummies['year'] = df_movies_dummies['production_date'].dt.year
df_movies_dummies['year'] = df_movies_dummies['year'].astype(int)

columns_to_drop = ['movie_title', 'director_name', 'director_professions', 'director_deathYear', 'director_birthYear', 'production_date']
df_movies_dummies = df_movies_dummies.drop(columns=columns_to_drop)


In [ ]:

features = ['runtime_minutes', 'movie_averageRating', 'movie_numerOfVotes', 
               'approval_Index', 'Production budget $', 'Domestic gross $', 'Worldwide gross $']

for feature in features:
    mean = df_movies_dummies[feature].mean()
    std = df_movies_dummies[feature].std()
    df_movies_dummies[feature] = (df_movies_dummies[feature] - mean) / std
    
df_movies_dummies.head()

X = df_movies_dummies[['runtime_minutes', 'movie_averageRating', 'movie_numerOfVotes', 
               'approval_Index', 'Production budget $', 'Domestic gross $', 'Worldwide gross $']]

y = df_movies_dummies[['Action', 'Adventure', 'Animation', 'Biography',
       'Comedy', 'Crime', 'Documentary', 'Drama', 'Family', 'Fantasy',
       'Film-Noir', 'History', 'Horror', 'Music', 'Musical', 'Mystery', 'News',
       'Romance', 'Sci-Fi', 'Sport', 'Thriller', 'War', 'Western']]

X = np.asarray(X)
y = np.asarray(y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20)

y_train

In [ ]:
X = np.asarray(X)
y = np.asarray(y)

X_train, X_test, y_train, y_test = train_test_split(
  X,
  y,
  test_size=0.20
)

In [ ]:
model = Sequential([
    Dense(256, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.5),
    BatchNormalization(),
    Dense(128, activation='relu',),
    Dropout(0.5),
    BatchNormalization(),
    Dense(64, activation='relu',),
    BatchNormalization(),
    Dense(32, activation='relu',),
    BatchNormalization(),
    Dense(y_train.shape[1], activation='sigmoid')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

early_stopping = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)

history1 = model.fit(X_train, y_train, epochs=200, batch_size=64, validation_data=(X_test, y_test), callbacks=[early_stopping])

test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Accuracy on test data: {test_accuracy}")

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'svg'
def plot_metric(history, metric):
    train_metrics = history.history[metric]
    val_metrics = history.history['val_'+metric]
    epochs = range(1, len(train_metrics) + 1)
    plt.plot(epochs, train_metrics)
    plt.plot(epochs, val_metrics)
    plt.title('Training and validation '+ metric)
    plt.xlabel("Epochs")
    plt.ylabel(metric)
    plt.legend(["train_"+metric, 'val_'+metric])
    plt.show()

plot_metric(history1, 'accuracy')

In [ ]:
model2 = Sequential([
    Dense(512, activation='sigmoid', input_shape=(X_train.shape[1],)),
    BatchNormalization(),
    Dropout(0.4),
    Dense(512, activation='sigmoid',kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    Dropout(0.4),
    Dense(256, activation='sigmoid', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    Dropout(0.4),
    Dense(128, activation='sigmoid', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    Dropout(0.4),
    Dense(64, activation='sigmoid', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    Dropout(0.4),
    Dense(32, activation='sigmoid', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    Dense(y_train.shape[1], activation='sigmoid')
])

model2.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

early_stopping = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)

history2 = model2.fit(X_train, y_train, epochs=200, batch_size=64, validation_data=(X_test, y_test), callbacks=[early_stopping])

test_loss, test_accuracy = model2.evaluate(X_test, y_test, verbose=0)
print(f"Accuracy on test data: {test_accuracy}")

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'svg'
def plot_metric(history, metric):
    train_metrics = history.history[metric]
    val_metrics = history.history['val_'+metric]
    epochs = range(1, len(train_metrics) + 1)
    plt.plot(epochs, train_metrics)
    plt.plot(epochs, val_metrics)
    plt.title('Training and validation '+ metric)
    plt.xlabel("Epochs")
    plt.ylabel(metric)
    plt.legend(["train_"+metric, 'val_'+metric])
    plt.show()

plot_metric(history2, 'accuracy')

In [ ]:
model3 = Sequential([
    Dense(1024, activation='tanh', input_shape=(X_train.shape[1],)),
    BatchNormalization(),
    Dropout(0.5),
    Dense(512, activation='tanh',kernel_regularizer=l1(0.001)),
    BatchNormalization(),
    Dropout(0.5),
    Dense(512, activation='tanh', kernel_regularizer=l1(0.001)),
    BatchNormalization(),
    Dropout(0.5),
    Dense(512, activation='tanh', kernel_regularizer=l1(0.001)),
    BatchNormalization(),
    Dropout(0.5),
    Dense(256, activation='tanh', kernel_regularizer=l1(0.001)),
    BatchNormalization(),
    Dropout(0.5),
    Dense(128, activation='tanh', kernel_regularizer=l1(0.001)),
    BatchNormalization(),
    Dropout(0.5),
    Dense(64, activation='tanh', kernel_regularizer=l1(0.001)),
    BatchNormalization(),
    Dropout(0.5),
    Dense(32, activation='tanh', kernel_regularizer=l1(0.001)),
    BatchNormalization(),
    Dense(y_train.shape[1], activation='sigmoid')
])

model3.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

early_stopping = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)

history3 = model3.fit(X_train, y_train, epochs=200, batch_size=64, validation_data=(X_test, y_test), callbacks=[early_stopping])

test_loss, test_accuracy = model3.evaluate(X_test, y_test, verbose=0)
print(f"Accuracy on test data: {test_accuracy}")

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'svg'
def plot_metric(history, metric):
    train_metrics = history.history[metric]
    val_metrics = history.history['val_'+metric]
    epochs = range(1, len(train_metrics) + 1)
    plt.plot(epochs, train_metrics)
    plt.plot(epochs, val_metrics)
    plt.title('Training and validation '+ metric)
    plt.xlabel("Epochs")
    plt.ylabel(metric)
    plt.legend(["train_"+metric, 'val_'+metric])
    plt.show()

plot_metric(history3, 'accuracy')